# Processing an FDN in Python

This notebook is a Python translation of `example_processFDN.m`.

It loads a dry example signal, configures a feedback delay network (FDN) with
random delays and a velvet feedback matrix, designs simple one‑pole
absorption filters, and renders a stereo reverberant output to disk.

In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess

from importlib.resources import files

import numpy as np
from scipy.io import wavfile

from pyFDN.auxiliary.delay import matrix_delay_approximation
from pyFDN.auxiliary.acoustics import one_pole_absorption
from pyFDN.auxiliary.filters import ZTF
from pyFDN.generate.construct_velvet_feedback_matrix import construct_velvet_feedback_matrix
from pyFDN.generate.random_matrix_shift import random_matrix_shift
from pyFDN.process import process_fdn

## Load and prepare the input signal

We read a short dry audio example from the `pyFDN` package, convert it to
floating point in the range [-1, 1], and append some silence so that the
reverb tail can decay fully.

In [2]:
audio_resource = files("pyFDN.audio") / "synth_dry.wav"
with audio_resource.open("rb") as audio_file:
    fs, audio = wavfile.read(audio_file)

# Use first channel if the file is stereo
x = audio[:, 0] if audio.ndim > 1 else audio

# Convert to float in [-1, 1]
if np.issubdtype(x.dtype, np.integer):
    max_val = np.iinfo(x.dtype).max
    x = x.astype(np.float64) / max_val
else:
    x = x.astype(np.float64)

# Append 2 seconds of silence for the decay tail
x = np.concatenate([x, np.zeros(2 * fs, dtype=np.float64)])

print(f"Sampling rate: {fs} Hz")
print(f"Input length:  {x.shape[0]} samples")

Sampling rate: 48000 Hz
Input length:  519872 samples


## Configure the FDN structure

We choose delay-line lengths, random input/output gains, and construct a
velvet feedback matrix with additional random shifts. We also compute a
matrix-delay approximation for diagnostics.

In [3]:
rng = np.random.default_rng(seed=0)

N = 8
num_input = 1
num_output = 2

input_gain = np.ones((N, num_input))
output_gain = rng.random((num_output, N))
direct = np.zeros((num_output, num_input))

# Delay-line lengths (in samples)
delays = rng.integers(500, 2001, size=N)

number_of_stages = 2
sparsity = 3
max_shift = 30

feedback_matrix, rev_feedback_matrix = construct_velvet_feedback_matrix(
    N, number_of_stages, sparsity
)
feedback_matrix, rev_feedback_matrix, _, _ = random_matrix_shift(
    max_shift, feedback_matrix, rev_feedback_matrix
)

approximation, approximation_error = matrix_delay_approximation(feedback_matrix)

print("Delay lengths (samples):", delays)
print("Approximation mean abs error:", np.mean(np.abs(approximation_error)))

Delay lengths (samples): [ 634 1795  533 1312  620  949 1222 1134]
Approximation mean abs error: 3.6240054408810085


## Design one-pole absorption filters

We use `one_pole_absorption` to design simple first-order IIR filters with
specified reverberation times at DC and Nyquist for each delay line.

In [4]:
rt_dc = 2.0   # target T60 at DC (seconds)
rt_ny = 0.5   # target T60 at Nyquist (seconds)

# Design one-pole absorption filters (SOS format: shape (6, N))
sos = one_pole_absorption(rt_dc, rt_ny, delays, fs=float(fs))

# Convert SOS to a diagonal ZTF filter matrix compatible with process_fdn
b0 = sos[0, :]
a1 = sos[4, :]

N = delays.shape[0]

b_abs = np.zeros((N, 1, 2), dtype=np.float64)
a_abs = np.zeros((N, 1, 2), dtype=np.float64)

b_abs[:, 0, 0] = b0

a_abs[:, 0, 0] = 1.0
a_abs[:, 0, 1] = a1

z_absorption = ZTF(b_abs, a_abs, is_diagonal=True)

print("Designed one-pole absorption filters.")

Designed one-pole absorption filters.


## Process the signal through the FDN

Finally we run the block-based FDN processor, normalise the output to avoid
clipping, save it as a WAV file, and (on macOS) play it back with `afplay`.

In [5]:
output = process_fdn(
    x[:, np.newaxis],
    delays,
    feedback_matrix,
    input_gain,
    output_gain,
    direct,
    absorption_filters=z_absorption,
)

output = np.asarray(output, dtype=np.float64)
if output.ndim == 1:
    output = output[:, np.newaxis]

peak = np.max(np.abs(output))
if peak > 0:
    output = output / peak * 0.99

outfile_path = Path.cwd() / "example_process_fdn_output.wav"
wavfile.write(outfile_path, fs, np.int16(output * 32767.0))

try:
    subprocess.run(["afplay", str(outfile_path)], check=False)
except FileNotFoundError:
    pass

print("Output shape:", output.shape)
print("Output RMS:", np.sqrt(np.mean(output ** 2)))
print("Wrote output to:", outfile_path)

Output shape: (519872, 2)
Output RMS: 0.12647615796201306
Wrote output to: /Users/wu12recu/Documents/GitHub/pyFDN/examples/example_process_fdn_output.wav
